# 조건 A/B 비교 보고서 템플릿

동일한 질문·평가 기준으로 만든 두 Repeatability 실행을 비교합니다. 저장된 수치와 조건만 표시합니다.


In [ ]:
import os

import mlflow
import panel as pn

from elice_reporting import export_html, find_root, from_files, from_mlflow, report_purpose_callout
from elice_reporting.comparison_sections import _selector
from elice_reporting.repeated_comparison_sections import (
    conditions,
    efficiency_view,
    failure_view,
    metric_guide,
    quality_view,
    question_view,
    stability_view,
    subject_info,
)

pn.extension()
ROOT = find_root()

In [ ]:
# 이 셀의 입력만 보고서마다 수정합니다. 인증정보는 환경변수에 둡니다.
SOURCE_MODE = "mlflow"  # mlflow | local
RUN_ID = "7ccaac47124543db82d3708a16c9c13e"
REPORT_TITLE = "GPT 6.1 Terra ↔ Qwen3.6-35B-A3B 비교"
OUTPUT_HTML = ROOT / "artifacts/reports/gpt-terra-vs-qwen-comparison.html"
LOCAL_REPORT = ROOT / "artifacts/evaluation/comparison-result.json"
LOCAL_INPUTS = ROOT / "artifacts/evaluation/comparison-inputs.json"

In [ ]:
if SOURCE_MODE == "mlflow":
    mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
    if not RUN_ID:
        raise ValueError("비교할 MLflow comparison RUN_ID를 설정하세요.")
    data = from_mlflow("comparison", RUN_ID, ROOT / "artifacts/reports/cache")
elif SOURCE_MODE == "local":
    data = from_files("comparison", LOCAL_REPORT, LOCAL_INPUTS)
else:
    raise ValueError("SOURCE_MODE는 mlflow, local 중 하나여야 합니다.")
if data.report.get("comparison_contract") != "comparison-v2":
    raise ValueError("Repeatability A/B 비교에는 comparison-v2 결과가 필요합니다.")

## 보고서 화면


In [ ]:
subjects = subject_info(data)
results = _selector(
    [
        ("경계 충족률", quality_view(data, subjects)),
        ("시행 간 변화폭", stability_view(data, subjects)),
        ("효율성 · 토큰/시간", efficiency_view(data)),
        ("생성 실패", failure_view(data)),
    ],
    "결과 범주",
)
report_view = pn.Column(
    pn.pane.Markdown("# " + REPORT_TITLE),
    report_purpose_callout("comparison"),
    pn.pane.Markdown("## 1. 평가 조건"),
    conditions(data, subjects),
    pn.pane.Markdown("## 2. 지표 정의와 판정 기준"),
    metric_guide(data),
    pn.pane.Markdown("## 3. 결과 분석"),
    results,
    pn.pane.Markdown("## 4. 문항별 비교"),
    question_view(data),
    sizing_mode="stretch_width",
    max_width=1100,
    styles={"margin": "0 auto"},
)
report_view

## 독립 HTML export


In [ ]:
html_path = export_html(report_view, OUTPUT_HTML, title=REPORT_TITLE, overwrite=True)
html_path